LeNet-5 모델 클래스 정의

In [ ]:
import torch
import torch.nn as nn

class LeNet5(nn.Module):
    def __init__(self, num_classes=10):
        super(LeNet5, self).__init__()

        # 첫 번째 합성곱 계층: Input (1, 32, 32) -> Output (6, 28, 28)
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5, stride=1)
        self.relu1 = nn.ReLU()
        # 첫 번째 평균 풀링 계층: Output (6, 14, 14)
        self.pool1 = nn.AvgPool2d(kernel_size=2, stride=2)

        # 두 번째 합성곱 계층: Output (16, 10, 10)
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, stride=1)
        self.relu2 = nn.ReLU()
        # 두 번째 평균 풀링 계층: Output (16, 5, 5)
        self.pool2 = nn.AvgPool2d(kernel_size=2, stride=2)

        # 완전 연결 계층 (Fully Connected Layers)
        # Flatten 후 크기: 16 * 5 * 5 = 400
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.relu3 = nn.ReLU()

        self.fc2 = nn.Linear(120, 84)
        self.relu4 = nn.ReLU()

        self.fc3 = nn.Linear(84, num_classes)

    def forward(self, x):
        # Feature Extraction
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))

        # Flatten
        x = torch.flatten(x, 1)

        # Classification
        x = self.relu3(self.fc1(x))
        x = self.relu4(self.fc2(x))
        x = self.fc3(x)
        return x

모델 인스턴스 생성 및 구조 확인

In [3]:
######################################################
# 1. 라이브러리 임포트 및 디바이스 설정
######################################################
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

########################################################
# 2. 데이터셋 로드 및 분할 (Data Preparation)
########################################################
# MNIST 기본 크기(28x28)를 LeNet5 입력 크기(32x32)에 맞게 리사이징
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

full_train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_size = 55000
val_size = 5000
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

##########################################################
# 3. 모델 생성, 손실 함수와 옵티마이저 설정
##########################################################
model = LeNet5(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

##########################################################
# 4. 학습 및 검증 루프 (Training & Validation Loop)
##########################################################
epochs = 10
for epoch in range(epochs):
    # --- [Training] ---
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()
        
    train_epoch_loss = train_loss / train_total
    train_epoch_acc = train_correct / train_total

    # --- [Validation] ---
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
            
    val_epoch_loss = val_loss / val_total
    val_epoch_acc = val_correct / val_total
    
    print(f"Epoch [{epoch+1}/{epochs}] | "
          f"Train Loss: {train_epoch_loss:.4f}, Train Acc: {train_epoch_acc*100:.2f}% | "
          f"Val Loss: {val_epoch_loss:.4f}, Val Acc: {val_epoch_acc*100:.2f}%")

##########################################################
# 5. 테스트 평가 (Test Evaluation)
##########################################################
model.eval()
test_loss, test_correct, test_total = 0.0, 0, 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        test_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        test_total += labels.size(0)
        test_correct += predicted.eq(labels).sum().item()

test_epoch_loss = test_loss / test_total
test_epoch_acc = test_correct / test_total
print(f"\n[Test Result] Loss: {test_epoch_loss:.4f}, Accuracy: {test_epoch_acc*100:.2f}%")

Using device: cpu


NameError: name 'transforms' is not defined